# Feature Inspection

This notebook lets you:
1. Run feature extraction and see the output
2. Load an existing `features.csv` and concat new patients onto it
3. Check NaN counts per subject and per feature
4. See exactly which patients are incomplete and what they're missing

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import gzip, shutil
import os

## Settings
Edit these paths to match your setup.

In [ ]:
MERGED_DIR     = Path("EOG_REM/merged_csv_eog_backup")      # !!! Change this to desired path or set to None to skip patient info
FEATURE_CSV    = Path("EOG_REM/features_csv/features.csv")  # !!! Change this to desired path or set to None to skip patient info
PATIENT_EXCEL  = Path("EOG_REM/path/to/patient_info.xlsx")  # !!! Change this to desired path or set to None to skip patient info
FS             = 250.0
PATTERN        = "*contiguous_eog_merged.csv"

---
## 1. Compress merged CSVs

Run this to compress all merged CSVs in `merged_csv_eog` and delete the uncompressed versions. This is optional but saves a lot of disk space. You can still read the compressed CSVs with `pd.read_csv(..., compression='gzip')`

In [ ]:
# compress_merged.py  (delete after running)

for f in Path("EOG_REM/merged_csv_eog").glob("*contiguous_eog_merged.csv"):
    gz = f.with_suffix(".csv.gz")
    with open(f, "rb") as fi, gzip.open(gz, "wb", 6) as fo:
        shutil.copyfileobj(fi, fo)
    saved = f.stat().st_size - gz.stat().st_size
    print(f"{f.name}: freed {saved / 1024**2:.1f} MB")
    f.unlink()

---
## 2. Size check
This cell checks the size of the merged CSVs before and after compression, and prints out how much space was saved for each file and in total. You can run this after the compression step to verify that it worked as expected.

In [ ]:
rows = []
for f in sorted(MERGED_DIR.glob("*.csv.gz")):
    size_mb = f.stat().st_size / 1e6
    try:
        df = pd.read_csv(f, usecols=["time_sec"], low_memory=True)
        duration_min = df["time_sec"].iloc[-1] / 60
        n_samples = len(df)
    except Exception as e:
        duration_min = None
        n_samples = None
    rows.append({
        "file": f.name,
        "size_mb": round(size_mb, 1),
        "n_samples": n_samples,
        "duration_min": round(duration_min, 1) if duration_min else None,
    })

df_sizes = pd.DataFrame(rows).sort_values("size_mb", ascending=False)
print(df_sizes.to_string(index=False))

# Flag subjects where lights trimming likely failed
THRESHOLD_MIN = 600  # 10 hours
df_suspects = df_sizes[df_sizes["duration_min"] > THRESHOLD_MIN].copy()
df_suspects["subject_id"] = df_suspects["file"].str.extract(r"(DCSM_\d+_[a-zA-Z])")

print(f"\nSubjects with recording > {THRESHOLD_MIN} min ({THRESHOLD_MIN/60:.0f} h):")
print(f"  {len(df_suspects)} / {len(df_sizes)}\n")
print(df_suspects[["subject_id", "duration_min"]].to_string(index=False))

---
## 3. NaN inspection
Load `features.csv` and check for missing values.

In [ ]:
df = pd.read_csv(FEATURE_CSV)
print(f"Loaded: {df.shape[0]} subjects, {df.shape[1]} columns")

### 3a. NaN count per feature
Which features have the most missing values?

In [ ]:
nan_per_feature = df.isna().sum()
nan_per_feature = nan_per_feature[nan_per_feature > 0].sort_values(ascending=False)

if nan_per_feature.empty:
    print("No NaN values in any feature!")
else:
    print(f"{len(nan_per_feature)} features have NaN values:\n")
    for feat, count in nan_per_feature.items():
        pct = count / len(df) * 100
        print(f"  {feat:<45s}  {count:3d} / {len(df)}  ({pct:.1f}%)")

### 3b. NaN count per subject
Which subjects have missing values, and how many?

In [ ]:
id_cols = ["subject_id", "DCSM_ID"]
feature_cols = [c for c in df.columns if c not in id_cols and pd.api.types.is_numeric_dtype(df[c])]

nan_per_subject = df[feature_cols].isna().sum(axis=1)
subjects_with_nan = df[nan_per_subject > 0][["subject_id"]].copy()
subjects_with_nan["nan_count"] = nan_per_subject[nan_per_subject > 0].values

if subjects_with_nan.empty:
    print("All subjects have complete features!")
else:
    print(f"{len(subjects_with_nan)} subject(s) have NaN values:\n")
    for _, row in subjects_with_nan.sort_values("nan_count", ascending=False).iterrows():
        print(f"  {row['subject_id']:<50s}  {row['nan_count']:.0f} missing features")

### 3c. Detailed NaN report
For each subject with NaN values, show exactly which features are missing.

In [ ]:
rows_with_nan = df[df[feature_cols].isna().any(axis=1)]

if rows_with_nan.empty:
    print("No NaN values found.")
else:
    for _, row in rows_with_nan.iterrows():
        sid = row["subject_id"]
        missing = [c for c in feature_cols if pd.isna(row[c])]
        print(f"\n{sid}  ({len(missing)} missing):")
        for col in missing:
            print(f"    - {col}")

### 3d. Group label check
Check which subjects have group labels (Control, iRBD, etc.) and which don't.

In [ ]:
group_cols = [c for c in ["Control", "PD(-RBD)", "PD(+RBD)", "iRBD", "PLM"] if c in df.columns]

if not group_cols:
    print("No group columns found. Run extract with --patient-excel to add them.")
else:
    has_label = df[group_cols].sum(axis=1) > 0
    n_labelled = has_label.sum()
    n_unlabelled = (~has_label).sum()
    
    print(f"With group label:     {n_labelled}")
    print(f"Without group label:  {n_unlabelled}")
    
    if n_unlabelled > 0:
        unlabelled = df.loc[~has_label, "subject_id"].tolist()
        print(f"\nUnlabelled subjects:")
        for sid in unlabelled:
            print(f"  - {sid}")
    
    # Group distribution
    print(f"\nGroup distribution:")
    for col in group_cols:
        n = (df[col] == 1).sum()
        print(f"  {col:<12s}  n = {n}")

---
## 4. Quick summary

In [ ]:
print(f"Total subjects:       {len(df)}")
print(f"Total features:       {len(feature_cols)}")
print(f"Subjects with NaN:    {len(rows_with_nan)}")
print(f"Features with NaN:    {len(nan_per_feature)}")
print(f"Total NaN values:     {df[feature_cols].isna().sum().sum()}")

---
## 5. ID cleanup

#### Function:

In [ ]:
def clean_feature_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    After collect_features + merge_patient_info, the DataFrame may contain:
      - subject_id  
      - DCSM_ID    
      - Duplicate group columns from patient module AND Excel join
        (Control, PD(-RBD), PD(+RBD), iRBD, PLM — possibly with _x/_y suffixes)

    This function consolidates duplicates and drops redundant columns.
    """
    df = df.copy()

    # 1) Drop DCSM_ID if it matches subject_id
    if "DCSM_ID" in df.columns:
        if df["DCSM_ID"].equals(df["subject_id"]):
            df.drop(columns=["DCSM_ID"], inplace=True)
            print("Dropped 'DCSM_ID' (identical to subject_id)")
        else:
            print("Kept 'DCSM_ID' (differs from subject_id for some rows)")

    # 2) Consolidate _x / _y suffix duplicates from pd.merge
    suffixed = {}
    for col in df.columns:
        for suffix in ("_x", "_y"):
            if col.endswith(suffix):
                base = col[: -len(suffix)]
                suffixed.setdefault(base, []).append(col)

    for base, cols in suffixed.items():
        # Keep _x, fill NaNs from _y, drop both suffixed versions
        primary = f"{base}_x"
        secondary = f"{base}_y"
        if primary in df.columns and secondary in df.columns:
            df[base] = df[primary].fillna(df[secondary])
            df.drop(columns=[primary, secondary], inplace=True)
            print(f"Merged '{primary}' + '{secondary}' → '{base}'")

    # 3) Drop any remaining _dup columns from the outer join
    dup_cols = [c for c in df.columns if c.endswith("_dup")]
    if dup_cols:
        df.drop(columns=dup_cols, inplace=True)
        print(f"Dropped _dup columns: {dup_cols}")

    print(f"\nFinal shape: {df.shape[0]} subjects × {df.shape[1]} columns")
    return df

#### Run function:

In [ ]:
df = pd.read_csv(FEATURE_CSV)                             # Load the original features CSV 
print(f"Before: {df.shape}")                              # Print shape before cleaning
df = clean_feature_columns(df)                            # Clean up redundant ID / label columns after merge
print(f"After: {df.shape}")                               # Print shape after cleaning
df.to_csv("features_csv/features_clean.csv", index=False) # Save the cleaned DataFrame to a new CSV file
print("Cleaned features saved to 'features_csv/features_clean.csv'")